In [ ]:
#HGSMGAT_Project
project_directory/
├── data/
│   ├── train.csv
│   └── test.csv
├── models/
│   └── custom_model.py
├── utils/
│   └── preprocessing.py
├── main.py
└── requirements.txt

In [1]:
pip install numpy pandas matplotlib seaborn scikit-learn shap dtw keras tensorflow networkx
pip install dtw-python
pip install shap
pip install networkx

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.manifold import SpectralEmbedding
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import shap
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, GRU, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load Data
train_path = 'ourdata/train.csv'
test_path = 'ourdata/test.csv'
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Combine for preprocessing
combined_df = pd.concat([train_df, test_df], axis=0)

# Identify numeric and categorical columns
numeric_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = combined_df.select_dtypes(exclude=[np.number]).columns.tolist()

# Impute missing values
num_imputer = SimpleImputer(strategy='mean')
cat_imputer = SimpleImputer(strategy='most_frequent')
combined_df[numeric_cols] = num_imputer.fit_transform(combined_df[numeric_cols])
combined_df[categorical_cols] = cat_imputer.fit_transform(combined_df[categorical_cols])

# Scale numeric features
scaler = MinMaxScaler()
combined_df[numeric_cols] = scaler.fit_transform(combined_df[numeric_cols])

# Split back into train and test
train_df = combined_df.iloc[:len(train_df), :]
test_df = combined_df.iloc[len(train_df):, :]

# Define features and target
features = [col for col in numeric_cols if col != 'target']
X = train_df[features].values
y = train_df['target'].values

# Compute DTW distance matrix
n_series = scaled_data.shape[1]
dtw_distances = np.zeros((n_series, n_series))

for i in range(n_series):
    for j in range(i + 1, n_series):
        dist, _, _, _ = accelerated_dtw(scaled_data[:, i], scaled_data[:, j], dist='euclidean')
        dtw_distances[i, j] = dist
        dtw_distances[j, i] = dist  # symmetric

# Spectral Embedding
embedding = SpectralEmbedding(n_components=10, affinity='nearest_neighbors')
X_embedded = embedding.fit_transform(X)

# Define custom GraphAttentionLayer
class GraphAttentionLayer(Layer):
    def __init__(self, output_dim, **kwargs):
        self.output_dim = output_dim
        super(GraphAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='kernel',
                                 shape=(input_shape[-1], self.output_dim),
                                 initializer='uniform',
                                 trainable=True)
        super(GraphAttentionLayer, self).build(input_shape)

    def call(self, x):
        return tf.matmul(x, self.W)

# Build Model
def build_model(input_shape):
    inputs = Input(shape=(input_shape,))
    x = GraphAttentionLayer(64)(inputs)
    x = GRU(32, return_sequences=False)(tf.expand_dims(x, axis=1))
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    return model

# Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_list, r2_list, mse_list, mae_list = [], [], [], []

for train_index, val_index in kf.split(X_embedded):
    X_train, X_val = X_embedded[train_index], X_embedded[val_index]
    y_train, y_val = y[train_index], y[val_index]

    model = build_model(X_train.shape[1])
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=100, batch_size=32, callbacks=[early_stop], verbose=0)

    y_pred = model.predict(X_val).flatten()
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)
    mse = mean_squared_error(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)

    rmse_list.append(rmse)
    r2_list.append(r2)
    mse_list.append(mse)
    mae_list.append(mae)

    # Plot training and validation loss
    plt.figure()
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

# Evaluation Metrics
print(f'Average RMSE: {np.mean(rmse_list):.4f}')
print(f'Average R²: {np.mean(r2_list):.4f}')
print(f'Average MSE: {np.mean(mse_list):.4f}')
print(f'Average MAE: {np.mean(mae_list):.4f}')

# Scatter Plot
plt.figure()
plt.scatter(y_val, y_pred)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted')
plt.show()

# Time Series Plot
plt.figure()
plt.plot(y_val, label='Actual')
plt.plot(y_pred, label='Predicted')
plt.xlabel('Sample')
plt.ylabel('Value')
plt.title('Time Series of Actual and Predicted')
plt.legend()
plt.show()

# SHAP Analysis
explainer = shap.DeepExplainer(model, X_train[:100])
shap_values = explainer.shap_values(X_val[:100])
shap.summary_plot(shap_values, X_val[:100], feature_names=features)

# Example municipal data
municipalities = ['Jaman North', 'Jaman South', 'Wenchi']
rmse_vals = [0.401, 0.327, 0.244]
r2_vals = [0.843, 0.860, 0.929]
mse_vals = [0.301, 0.287, 0.348]
mae_vals = [0.376, 0.364, 0.311]

# Create DataFrame
perf_df = pd.DataFrame({
    'Municipality': municipalities,
    'RMSE': rmse_vals,
    'R²': r2_vals,
    'MSE': mse_vals,
    'MAE': mae_vals
})

# Melt for plotting
perf_melted = perf_df.melt(id_vars='Municipality', var_name='Metric', value_name='Value')

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=perf_melted, x='Municipality', y='Value', hue='Metric')
plt.title('Municipal Performance Metrics')
plt.ylabel('Score')
plt.show()

